# 05 — Serving Cassandra / AstraDB

Carga del mart Gold `org_daily_usage_by_service` y ejecución de consultas #1 y #2.

**Requisitos:**
- Token de aplicación Astra (`ASTRA_DB_APPLICATION_TOKEN`)
- Secure connect bundle (`ASTRA_DB_SECURE_BUNDLE_PATH`)
- Keyspace `cloud_analytics` (se crea vía DDL o consola Astra)

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if IN_COLAB:
    !pip install -q pyspark cassandra-driver
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    # os.environ["ASTRA_DB_APPLICATION_TOKEN"] = "AstraCS:..."
    # os.environ["ASTRA_DB_SECURE_BUNDLE_PATH"] = "/content/secure-connect.zip"

from src.config import CASSANDRA_KEYSPACE, CQL_DIR
from src.cassandra.client import is_cassandra_configured

print(f"KEYSPACE: {CASSANDRA_KEYSPACE}")
print(f"CQL_DIR:  {CQL_DIR}")
print(f"Configured: {is_cassandra_configured()}")

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("serving-cassandra").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [ ]:
from src.jobs.serving_cassandra import dry_run_preview

preview = dry_run_preview(spark)
preview

In [ ]:
from src.cassandra.client import get_cassandra_session, read_cql_file, execute_cql_script
from src.config import CASSANDRA_KEYSPACE

session, cluster = get_cassandra_session()
execute_cql_script(session, read_cql_file("01_keyspace.cql"))
session.set_keyspace(CASSANDRA_KEYSPACE)
execute_cql_script(session, read_cql_file("02_org_daily_usage_by_service.cql"))
print("DDL OK")

In [ ]:
from src.jobs.serving_cassandra import load_org_daily_usage_by_service

org_id = preview["suggested_org_id_for_queries"]
load_result = load_org_daily_usage_by_service(spark, session)
load_result

In [ ]:
from src.jobs.serving_cassandra import (
    run_query_daily_costs_and_requests,
    run_query_top_services_by_cost,
)
import pandas as pd

# Consulta #1
q1 = run_query_daily_costs_and_requests(session, org_id=org_id)
display(pd.DataFrame(q1).head(10))

# Consulta #2 (top 5 servicios por costo acumulado, últimos 14 días)
q2 = run_query_top_services_by_cost(session, org_id=org_id, top_n=5)
display(pd.DataFrame(q2))

In [ ]:
# Idempotencia: re-cargar no debe duplicar filas
load_result_2 = load_org_daily_usage_by_service(spark, session)
assert load_result_2["count_after"] == load_result["count_after"]
assert load_result_2["idempotent_ok"]
print("Idempotencia OK")
cluster.shutdown()